# Prompt Optimization Techniques — AWS Bedrock Edition
A/B testing and iterative refinement using `boto3` + Amazon Nova on Bedrock.

**Architecture:**
- `nova-lite` — generates responses (fast, cheap)
- `nova-pro` — judges responses (smarter, more consistent)
- Majority vote (3 runs) — prevents random judge flipping
- Position bias fix — alternates response order each run

## Setup

In [ ]:
import re
import json
import boto3

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME  = "us.amazon.nova-lite-v1:0"  # generates responses (fast + cheap)
JUDGE_MODEL = "us.amazon.nova-pro-v1:0"   # judges responses  (smarter + consistent)
AWS_REGION  = "us-east-1"

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)
print(f"Generator : {MODEL_NAME}")
print(f"Judge     : {JUDGE_MODEL}")

## Core Helper — Bedrock Invocation

In [ ]:
def get_completion(prompt, system="", model_name=MODEL_NAME):
    """Invoke a Bedrock model (Nova or Claude) and return the text response.

    Args:
        prompt (str): The user-turn text.
        system (str): Optional system prompt.
        model_name (str): Bedrock model ID.

    Returns:
        str: The model response text.
    """
    if model_name.startswith("anthropic.claude"):
        body = json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 2000,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.0,
            "top_p": 1,
            "system": system,
        })
        response = bedrock.invoke_model(body=body, modelId=model_name)
        response_body = json.loads(response["body"].read())
        return response_body["content"][0]["text"]

    elif "amazon.nova" in model_name:
        body_dict = {
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {
                "max_new_tokens": 2000,
                "temperature": 0.0,
                "top_p": 1,
            },
        }
        if system:
            body_dict["system"] = [{"text": system}]

        response = bedrock.invoke_model(body=json.dumps(body_dict), modelId=model_name)
        response_body = json.loads(response["body"].read())
        return response_body["output"]["message"]["content"][0]["text"]

    else:
        raise ValueError(f"Unsupported model: {model_name}")


# Generator alias — uses MODEL_NAME (nova-lite)
def generate_response(prompt):
    return get_completion(prompt, model_name=MODEL_NAME)


# Replaces LangChain PromptTemplate.format()
def format_prompt(template, topic):
    return template.replace("{topic}", topic)

## Prompt Templates & Config
- **Prompt A** — intentionally weak (vague, no structure)
- **Prompt B** — intentionally strong (structured, audience-aware)
- **Topic** — niche enough that prompt quality actually changes output quality

In [ ]:
# Intentionally weak
PROMPT_A = "Tell me about {topic}."

# Intentionally strong
PROMPT_B = (
    "You are an expert teacher. Explain {topic} to a complete beginner. "
    "Structure your response with: (1) a simple one-line definition, "
    "(2) how it works step by step, (3) a real-world analogy, "
    "(4) a concrete example, (5) common misconceptions. "
    "Use plain language throughout."
)

TOPIC    = "transformer attention mechanism"
CRITERIA = ["clarity", "informativeness", "engagement", "accuracy", "completeness"]

## Head-to-Head Judge
- Uses **Nova Pro** as judge for smarter, more consistent verdicts
- Runs **3 times** and takes majority vote — prevents random flipping
- **Alternates response order** on even runs to cancel position bias

In [ ]:
def head_to_head(response_a, response_b, criteria, runs=3, judge_model=JUDGE_MODEL):
    """Compare two responses using a stronger judge model with majority voting.

    Args:
        response_a (str): Response from Prompt A.
        response_b (str): Response from Prompt B.
        criteria (list[str]): Evaluation criteria.
        runs (int): Number of judge runs for majority vote.
        judge_model (str): Bedrock model ID to use as judge.

    Returns:
        str: 'A' or 'B'
    """
    criteria_str = ", ".join(criteria)
    votes = []

    for i in range(1, runs + 1):
        print(f"  Judge run {i}/{runs} [{judge_model}]...")

        # Alternate order on even runs to cancel position bias
        if i % 2 == 0:
            first, second, label_first, label_second = response_b, response_a, "B", "A"
        else:
            first, second, label_first, label_second = response_a, response_b, "A", "B"

        comparison_prompt = (
            f"You are an expert evaluator. Compare these two responses on: {criteria_str}.\n\n"
            f"Response {label_first}:\n{first}\n\n"
            f"Response {label_second}:\n{second}\n\n"
            f"For each criterion, state which response is better and why.\n"
            f"Finally, output a single line exactly like this:\n"
            f"WINNER: A  or  WINNER: B"
        )

        result = get_completion(comparison_prompt, model_name=judge_model)

        match = re.search(r"WINNER:\s*([AB])", result)
        if match:
            votes.append(match.group(1))
            print(f"    Vote: {match.group(1)}")
        else:
            print(f"    Warning: could not extract vote. Skipping.")

    if not votes:
        print("No valid votes. Defaulting to A.")
        return "A"

    winner = "A" if votes.count("A") >= votes.count("B") else "B"
    print(f"\n  Votes → A: {votes.count('A')}  B: {votes.count('B')}")
    print(f"  Majority winner: {winner}")
    return winner

## A/B Test

In [ ]:
print("Generating responses with Nova Lite...\n")
response_a = generate_response(format_prompt(PROMPT_A, TOPIC))
response_b = generate_response(format_prompt(PROMPT_B, TOPIC))

print("=== A/B Test — Head-to-Head (Nova Pro judge) ===\n")
winner = head_to_head(response_a, response_b, CRITERIA)

print(f"\n✅ Winning prompt: {winner}")
best_template = PROMPT_A if winner == "A" else PROMPT_B

## Iterative Refinement
Takes the winning prompt and improves it over N rounds.
Strict output rules prevent Nova from returning a document instead of a clean prompt string.

In [ ]:
def refine_prompt(initial_template, topic, iterations=3):
    """Iteratively improve a prompt template using LLM feedback.

    Args:
        initial_template (str): Starting prompt template ({topic} as placeholder).
        topic (str): Subject to explain.
        iterations (int): Number of refinement rounds.

    Returns:
        str: Final refined prompt template.
    """
    current_template = initial_template

    for i in range(1, iterations + 1):
        print(f"\n--- Refinement iteration {i} ---")

        try:
            response = generate_response(format_prompt(current_template, topic))
        except Exception as e:
            print(f"  Error: {e}. Skipping iteration.")
            continue

        # Step 1: get feedback on the response
        feedback_prompt = (
            f"Analyze the following explanation of {topic} and suggest concrete "
            f"improvements to the prompt that generated it:\n\n{response}"
        )
        feedback = generate_response(feedback_prompt)

        # Step 2: rewrite the prompt — strict rules prevent document-style output
        refine_instruction = (
            f"Based on this feedback:\n'{feedback}'\n\n"
            f"Rewrite and improve the following prompt template.\n\n"
            f"STRICT RULES:\n"
            f"- Output ONLY the raw prompt string. Nothing else.\n"
            f"- No headers, no bullet points, no markdown, no explanations.\n"
            f"- No preamble like 'Here is the improved prompt:'\n"
            f"- Must contain the placeholder {{topic}} exactly once.\n"
            f"- Should be 1-4 sentences max.\n\n"
            f"Current template to improve:\n{current_template}"
        )
        refined_template = generate_response(refine_instruction)

        # Strip markdown fences and surrounding quotes
        refined_template = re.sub(r"```[^\n]*\n?", "", refined_template).strip()
        refined_template = refined_template.strip('"').strip("'")

        current_template = refined_template
        preview = current_template[:200]
        print(f"  New template:\n  {preview}{'...' if len(current_template) > 200 else ''}")

    return current_template

In [ ]:
refined_template = refine_prompt(best_template, TOPIC, iterations=3)

print("\n=== Final Refined Prompt ===")
print(refined_template)

## Compare Original vs Refined — Head-to-Head (Nova Pro judge)

In [ ]:
print("Generating responses for comparison...\n")
original_response = generate_response(format_prompt(best_template, TOPIC))
refined_response  = generate_response(format_prompt(refined_template, TOPIC))

print("=== Original vs Refined — Head-to-Head (Nova Pro judge) ===\n")
final_winner = head_to_head(original_response, refined_response, CRITERIA)

print(f"\n✅ Winner: {'Original' if final_winner == 'A' else 'Refined'}")
if final_winner == "B":
    print("Refinement improved the prompt!")
else:
    print("Original held up — try more iterations or a different topic.")